<a href="https://colab.research.google.com/github/Rogerio-mack/Modelos-de-Linguagem-e-Generativos-2026S1/blob/main/Modelo_Causal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Causal $\times$ Masked — os dois grandes paradigmas de Atenção

Vamos lembrar da arquitetura básica dos transformers:

![image](https://d2l.ai/_images/encoder-decoder.svg)

<br>



### **Causal LM** (ou *autoregressive*)

O modelo só pode "ver" os tokens **anteriores** ao token atual — nunca os seguintes. A atenção é **mascarada para o futuro**. Isso é implementado com uma *causal mask* (máscara triangular inferior) na atenção.

```
Token:   [O]  [gato]  [sentou]  [no]  [tapete]
                         ↑
              só vê "O" e "gato"
```

É o paradigma dos modelos **generativos** — GPT, LLaMA, Qwen, DeepSeek. O treinamento consiste em prever o próximo token, dado o contexto anterior. No HuggingFace eles são carregados com a classe `CausalLM`.




### **Masked LM** (ou *bidirectional*)

O modelo vê o contexto **inteiro** — passado e futuro — mas alguns tokens são mascarados e ele precisa adivinhá-los. É o paradigma do BERT.

```
Token:   [O]  [gato]  [MASK]  [no]  [tapete]
                         ↑
              vê todos os outros tokens
```

Ótimo para **entender** texto (classificação, NER, similaridade), mas não para gerar. Classe no HuggingFace: `MaskedLM`.



### **Seq2Seq** (encoder-decoder)

Um meio-termo: o encoder é bidirecional (lê tudo), o decoder é causal (gera token a token). Paradigma do T5, BART, mT5. Classe: `Seq2SeqLM`.



## Qual modelo empregar?

A escolha do paradigma determina **para que o modelo serve**:

| Paradigma | Arquitetura | Uso típico | Exemplos |
|---|---|---|---|
| Causal | Decoder-only | Geração de texto, chat, código | GPT, LLaMA, Qwen, DeepSeek |
| Masked | Encoder-only | Classificação, embeddings, NER | BERT, RoBERTa, MiniLM |
| Seq2Seq | Encoder-Decoder | Tradução, sumarização, Q&A | T5, BART, mT5 |

E determina **qual classe usar** no HuggingFace:

```python
# Geração (causal)
AutoModelForCausalLM.from_pretrained(...)

# Classificação / embeddings (masked)
AutoModel.from_pretrained(...)

# Tradução / sumarização (seq2seq)
AutoModelForSeq2SeqLM.from_pretrained(...)
```

**CUIDADO**: Usar a classe errada para o paradigma errado é um erro comum — o modelo carrega, mas o comportamento é incorreto ou sem sentido.



# Exemplo 1

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
device = "cuda" # the device to load the model onto

model = AutoModelForCausalLM.from_pretrained(
    "lazarohurtado/Qwen1.5-0.5B-OpenIT",
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("lazarohurtado/Qwen1.5-0.5B-OpenIT")

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt") # .to(device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

response

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


"The Large Language Model (LLM) is a subtask of the Transformer architecture, a pre-trained deep learning model that can accurately learn and convert sentences into machine-readable human-readable data. It is designed to provide an approximation of vocabulary size for tasks such as natural language understanding, sentence prediction, and dialogue systems. The LLM is trained by iteratively adjusting hyperparameters and learning from the training data. It is also available as a library in Python.\n\nThe LLM was initially developed by Yarrow Chen and Giulio Verma in the late 2010's as part of the TensorFlow and PyTorch collaborations. In 2017, the LLM was part of the Google Brain research workshop and is now considered a commonly used machine translation system in popular language development programs. The LLM's performance on the MNIST dataset has been consistently shown to be significantly worse compared to state-of-the-art systems. However, it remains a viable language translation tool

# Exemplo 2, em Português

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Qwen2.5-0.5B-Instruct: modelo causal com ~500M parâmetros
# Suficiente para o Colab gratuito (GPU T4 ou até CPU)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# O tokenizer converte texto em tokens (IDs numéricos)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# AutoModelForCausalLM: classe correta para modelos causais/generativos
# ... torch_dtype=auto: usa float16 se houver GPU, float32 em CPU
# ... device_map="auto": distribui o modelo entre GPU e CPU automaticamente
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## Modelos "Instruct"

Modelos "Instruct" esperam o prompt em formato de chat (lista de mensagens). Eles não aceitam texto puro — eles esperam tokens especiais que marcam início e fim de cada questão (como <|im_start|>user). Cada família de modelos tem seu próprio formato; apply_chat_template cuida disso automaticamente lendo o template salvo junto ao tokenizer.


In [13]:
# ... "system": instrui o comportamento geral do modelo
# ... "user": a pergunta ou instrução do usuário
messages = [
    {"role": "system", "content": "Você é um assistente prestativo. Responda em português."},
    {"role": "user",   "content": "O que é dengue?"},
]

# apply_chat_template: formata as mensagens no template exato do modelo
#
# Cada modelo instruct tem seu próprio template (tokens especiais, separadores)
# tokenize=False → retorna string; add_generation_prompt=True → prepara para geração
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# Tokeniza a string formatada e envia para o device correto (GPU ou CPU)
inputs = tokenizer([text], return_tensors="pt") # .to(model.device)

In [16]:
print(text)

<|im_start|>system
Você é um assistente prestativo. Responda em português.<|im_end|>
<|im_start|>user
O que é dengue?<|im_end|>
<|im_start|>assistant



In [17]:
print(inputs)

{'input_ids': tensor([[151644,   8948,    198,  69286,   3958,   4443,   7789,   6817,    855,
           9878,   6496,     13,   1800,    618,   3235,    976,   2635,  29785,
          36830,     13, 151645,    198, 151644,    872,    198,     46,   1709,
           3958,    294,    826,    361,     30, 151645,    198, 151644,  77091,
            198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [14]:
# model.generate(): loop autoregressivo — gera um token por vez
# ... max_new_tokens: limita o tamanho da resposta (não do prompt)
# ... do_sample=True: amostragem estocástica (mais criativo que greedy)
# ... temperature: controla aleatoriedade (0=determinístico, 1=padrão, >1=mais caótico)
# ... top_p: nucleus sampling — considera só tokens que somam 90% de probabilidade
output_ids = model.generate(
    **inputs,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)

# Remove os tokens do prompt da saída (só queremos a resposta gerada)
# output_ids[0] é o primeiro (e único) item do batch
generated_ids = output_ids[0][inputs.input_ids.shape[1]:]

# decode: converte IDs numéricos de volta para texto legível
# skip_special_tokens=True: remove tokens como <|endoftext|>, <|im_end|> etc.
response = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(response)

Dengue é uma doença transmitteda pelo mosquito de Aedes aegypti, o mosquito comum do ar no hemisfério sul do mundo. Esses insetos têm longas ecurias, causando infecções cutâneas e circulares no corpo humano.


In [18]:
print(generated_ids)

tensor([    35,    826,    361,   3958,  10608, 138716,  33599,     64,  27525,
         49546,    409,    362,  58526,    264,    791,     88,    417,     72,
            11,    297,  49546,    469,    372,    653,    796,    902,  17280,
           285,  58858,  10383,  25774,    653,  28352,     13,  12838,    288,
         54859,    436,  98860,   1293,    300,    384,   2352,   3473,     11,
         24524,   4883,   4132,    757,  15249,   3931,   8835,  97257,    384,
          4225,    360,   5403,    902,  65634,  96457,     13, 151645])
